### Ingestion
Here we will follow below mnetioned steps:

1. Read the mapping sheet and create the table w.r.t it

2. add the all the SCD2 logic on each table
    - MERGE command based
    - APPLY SCD
    - APPLY Changes
    - using the haskey based logic

3. add the audit column and config column:

    Audit column:

    -     add_date
    -     delete_date
    -     last_modified_date
    -     is_active
    -     is_incremental
    -     hashkey

    Config column:
    -     data_source_id
    -     data_source
    -     dataset
    -     data_version

    


In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import pipelines as dp
import logging
import pandas as pd
import json,csv
import ast
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force =True
)

logger = logging.getLogger(__name__)

In [0]:
TYPE_MAP = {"varchar": "string", "string": "string", "int": "int",
            "bigint": "bigint", "double": "double", "date": "date",
            "timestamp": "timestamp", "boolean": "boolean"}

In [0]:
def read_mapping(map_path):
    # add .option("sep", "\t") if the sheet is tab-separated
    return spark.read.option("header", True).csv(map_path)

In [0]:
def parse_inc_arg(text):
    if not text or not text.strip():
        return {}
    return ast.literal_eval(text.strip())

In [0]:
def build_ingestion_df(map_df, cdm_table, src_catalog, src_schema):
    rules = (map_df
             .filter(col("is_active_map").cast("int") == 1)
             .filter(lower(trim("cdm_table")) == cdm_table.lower())
             .collect())
    if not rules:
        raise ValueError(f"No active mapping rows for {cdm_table!r}")

    src_table = rules[0]["src_table"].strip()
    # `rules` is a list of Rows, not a DataFrame
    target_types = sorted({r["target_type"].strip().lower()
        for r in rules
        if r["target_type"] and r["target_type"].strip()})
    # mapping holds the short source name; staging tables are named staging_<...>
    # prefer the staging tables: in the consolidated dev schema the bare src_table
    # (e.g. dev.provider) also exists as the moved dim table and would shadow the
    # real staging source, so it is only a last-resort fallback
    src_candidates = [f"staging_{src_table}", f"staging_{cdm_table}", src_table]
    src_table_name = next(
        (f"{src_catalog}.{src_schema}.{c}" for c in src_candidates
         if spark.catalog.tableExists(f"{src_catalog}.{src_schema}.{c}")),
        f"{src_catalog}.{src_schema}.{src_table}",
    )

    spark.sql(f"""
    ALTER TABLE {src_table_name}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true);          
    """)
    if "fact" in target_types:
        src_df = (spark.readStream
            .option("readChangeFeed", "true")
            .table(src_table_name)
            .filter(col("_change_type") != "update_preimage"))
    else:
        src_df = spark.read.table(src_table_name)
    
    src_name = (rules[0]["src_name"] or "").strip()
    value_defaults = {
        "data_source_id": src_name.upper(),
        "data_source": src_name,
        "dataset": src_table,
    }

    exprs, key_cols, non_key_cols = [], [], []
    for r in rules:
        tgt      = r["cdm_column"].strip()
        src_col  = (r["src_column"] or "").strip()
        sql_tx   = (r["sql_tx"] or "").strip()
        dtype    = TYPE_MAP.get((r["data_type"] or "string").strip().lower(), "string")
        if dtype == "int":
            # widen to bigint: provider_id holds 10-digit values that overflow INT
            dtype = "bigint"

        if sql_tx:
            col_expr = expr(sql_tx)
        elif src_col:
            col_expr = col(f"`{src_col}`")
        elif tgt in value_defaults:
            col_expr = lit(value_defaults[tgt])
        else:
            raise ValueError(f"{cdm_table}.{tgt}: no src_column, sql_tx or default value")

        if dtype == "date":
            # staging dates arrive as MM/dd/yyyy strings, not ISO yyyy-MM-dd
            exprs.append(to_date(col_expr, "MM/dd/yyyy").alias(tgt))
        else:
            exprs.append(col_expr.cast(dtype).alias(tgt))
        (key_cols if (r["map_type"] or "").strip().upper() == "PRIMARY" else non_key_cols).append(tgt)

    df = src_df.select(*exprs)

    # inc_arg appears only on the first row of each table in your sheet
    inc = {}
    for r in rules:
        if not r["inc_arg"]:
            continue
        try:
            inc = parse_inc_arg(r["inc_arg"])
        except (ValueError, SyntaxError):
            inc = {}
        if isinstance(inc, str):
            # Spark's CSV reader doubles the quotes around values, so the dict
            # comes back as a string with unquoted values: {'hash_key_reqd':Y}
            try:
                inc = ast.literal_eval(inc.replace(":Y}", ":'Y'}"))
            except (ValueError, SyntaxError):
                inc = {}
        if isinstance(inc, dict) and inc:
            break
    opts = (inc.get("EXCEPT_KEY_COLUMN_CHECKSUM_LOAD")
        or inc.get("ALL_COLUMN_CHECKSUM_LOAD")
        or {})
    if opts.get("hash_key_reqd") == "Y" and non_key_cols:
        df = df.withColumn("row_hash", sha2(concat_ws("||",
             *[coalesce(col(c).cast("string"), lit("<null>")) for c in non_key_cols]), 256))

    scd = (rules[0]["scd"] or "").strip().upper() == "Y"
    return df, key_cols, scd,target_types

In [0]:
def SCD_UPSERT_logic(df,key,target):
    "here we will apply the SCD with the upsert logic"
    view = f"ing_{target.replace('.', '_')}"
    df.createOrReplaceTempView(view)

    # columns with special handling in the MERGE
    audit_cols = {"hashkey", "row_hash", "add_date", "delete_date",
                  "last_modified_date", "is_active", "is_incremental"}

    # keys arrive as a list from build_ingestion_df; accept a single name too
    keys = list(key) if isinstance(key, (list, tuple)) else [key]

    # everything else in the dataframe is copied as-is
    value_cols = [c for c in df.columns if c not in keys and c not in audit_cols]

    if "row_hash" not in df.columns:
        raise ValueError(f"{target}: source dataframe has no row_hash column")

    set_vals = ",\n".join(f"{c} = src.{c}" for c in value_cols)
    ins_cols = ", ".join([*keys, *value_cols])
    ins_vals = ", ".join([f"src.{k}" for k in keys] + [f"src.{c}" for c in value_cols])
    on_expr  = " and ".join(f"tgt.{k} = src.{k}" for k in keys)

    # first run: create the target table with the audit columns
    if not spark.catalog.tableExists(target):
        (df
         .withColumnRenamed("row_hash", "hashkey")
         .withColumn("add_date", current_timestamp())
         .withColumn("delete_date", to_timestamp(lit("2099-12-31 00:00:00")))
         .withColumn("last_modified_date", current_timestamp())
         .withColumn("is_active", lit(1))
         .withColumn("is_incremental", lit(1))
         .write.format("delta").mode("overwrite").saveAsTable(target))
        logger.info(f"{target}: initial full load")
        return None

    # audit-column semantics:
    #   not matched (new row)              -> fresh audit columns, is_incremental = 1
    #   matched, hash differs (changed row) -> hashkey, add_date, delete_date, last_modified_date
    #                                         all refreshed; is_active = 1; is_incremental = 1
    #   matched, hash identical (unchanged)  -> only is_active = 1 and is_incremental = 0;
    #                                         add_date/delete_date/last_modified_date untouched
    dim_df = spark.sql(f"""
    MERGE INTO {target} as tgt
    using {view} as src
    on {on_expr}
    when matched and tgt.hashkey <> src.row_hash
    then update set 
    {set_vals},
    tgt.hashkey = src.row_hash,
    tgt.add_date = current_timestamp(),
    tgt.delete_date = '2099-12-31 00:00:00',
    tgt.last_modified_date = current_timestamp(),
    tgt.is_active = 1,
    tgt.is_incremental = 1
    when matched and tgt.hashkey = src.row_hash
    then update set 
    tgt.is_active = 1,
    tgt.is_incremental = 0
    when not matched
    then insert ({ins_cols}, 
    hashkey,
    add_date, 
    delete_date, 
    last_modified_date, 
    is_active, 
    is_incremental)
    VALUES ({ins_vals}, 
    src.row_hash,
    current_timestamp(), 
    '2099-12-31 00:00:00', 
    current_timestamp(), 
    1, 
    1)""")
    return dim_df


In [0]:
import os

TEMPLATE_PATH = "/Volumes/dab/healthcare_datasets/healthcare_vol/sql/SCD2_fact.sql"
OUT_DIR       = "/Workspace/pipelines/sql"      # where the pipeline reads its source files

def scd_cdc_logic(df, src, key_cols, target, scd_type=2, seq_col="last_modified_date"):
    """Render the AUTO CDC SQL template for one table and write it out."""
    # columns handled by the template itself
    special = {"hashkey", "add_date", "delete_date",
               "last_modified_date", "is_active", "is_incremental"}
    # the template re-exposes row_hash as hashkey, so keep it out of {cols}
    value_cols = [c for c in df.columns
                  if c not in key_cols and c not in special and c != "row_hash"]
    with open(TEMPLATE_PATH) as f:
        template = f.read()

    sql = template.format(
        src=src,                                   # upstream table, e.g. cat.schema.fact_claims
        target=target,                             # short name, e.g. fact_claims
        cols=", ".join(f"`{c}`" for c in [*key_cols, *value_cols]),
        keys=", ".join(f"`{c}`" for c in key_cols),
        seq_col=seq_col,
        scd_type=scd_type,
        track="TRACK HISTORY ON hashkey" if scd_type == 2 else "",
    )
    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = f"{OUT_DIR}/{target}.sql"
    with open(out_path, "w") as f:
        f.write(sql)
    return out_path

In [0]:
def create_ingestion_table(config, src_table,cdm_table, layer="ingestion", src_layer="staging"):
    cfg, src = config[layer], config[src_layer]
    map_df = read_mapping(cfg["mappath"])

    df, keys, scd, target_types = build_ingestion_df(map_df, cdm_table, src["catalog"], src["schema"])
    if not df.isStreaming:                # dropDuplicates needs a watermark on streams
        df = df.dropDuplicates(keys)      # one row per key

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{cfg['catalog']}`.`{cfg['schema']}`")
    target = f"{cfg['catalog']}.{cfg['schema']}.ingestion_{cdm_table}"
    source = f"{cfg['catalog']}.{cfg['schema']}.{src_table}"
    is_dim = any(t in ("dim", "dimension") for t in target_types)

    if scd and is_dim:
        # full-load dimension: batch MERGE, runs in this notebook
        SCD_UPSERT_logic(df, keys, target)
    elif scd:
        # incremental fact: land the transformed rows, then generate the pipeline SQL
        if df.isStreaming:
            # facts get a CDF readStream source: land them with a one-shot stream
            # write sharing the fact checkpoint, so it resumes where it left off
            (df.writeStream
               .option("checkpointLocation",
                       f"/Volumes/dab/healthcare_datasets/healthcare_vol/data/checkpoint/{cdm_table}")
               .outputMode("append")
               .trigger(availableNow=True)
               .toTable(target)
               .awaitTermination())
        else:
            df.write.mode("append").saveAsTable(target)
        # the pipeline SQL maintains SCD2 history on top of the rows this notebook
        # lands in the ingestion table (the landing is append-only, so it streams
        # cleanly); the 4th arg is the SHORT name the template uses to build
        # src_<t>/scd_<t>/<t> objects, not the full path
        sql_path = scd_cdc_logic(df, target, keys, cdm_table)
        logger.info(f"{cdm_table}: generated {sql_path}; run the pipeline to load {target}")

    elif is_dim:
        df.write.format("delta").mode("overwrite").saveAsTable(target)

    else:
        # non-SCD fact: append
        if df.isStreaming:
            (df.writeStream
               .option("checkpointLocation",
                       f"/Volumes/dab/healthcare_datasets/healthcare_vol/data/checkpoint/{cdm_table}")
               .outputMode("append")
               .trigger(availableNow=True)
               .toTable(target)
               .awaitTermination())
        else:
            df.write.format("delta").mode("append").saveAsTable(target)

    logger.info(f"Processed {target}")
    return df

In [0]:
config_path = '/Volumes/dab/healthcare_datasets/healthcare_vol/config/dev_config.json'
with open(config_path) as f:
    config = json.load(f)

ing_mappath = config["ingestion"]["mappath"]

tables = (read_mapping(ing_mappath)
          .filter(col("is_active_map").cast("int") == 1)
          .groupBy(trim("cdm_table").alias("cdm_table"),
                   trim("src_table").alias("src_table"))
          .agg(max(lower(trim("target_type"))).alias("target_type"))
          .collect())

# dimensions first, then facts
tables = sorted(tables, key=lambda r: 0 if r["target_type"] in ("dim", "dimension") else 1)

failed = []
for r in tables:
    try:
        create_ingestion_table(config, r["src_table"], r["cdm_table"])
    except Exception as e:
        logger.error(f"{r['cdm_table']} failed: {e}")
        failed.append(r["cdm_table"])

if failed:
    raise RuntimeError(f"Failed tables: {failed}")

2026-10-25 15:04:49,460 | INFO | __main__ | Processed devproduct.dev.ingestion_provider
2026-10-25 15:04:58,330 | INFO | __main__ | Processed devproduct.dev.ingestion_patient
2026-10-25 15:05:06,866 | INFO | __main__ | Processed devproduct.dev.ingestion_procedure_code
2026-10-25 15:05:15,508 | INFO | __main__ | Processed devproduct.dev.ingestion_diagnosis_code
2026-10-25 15:05:23,583 | INFO | __main__ | mx_claim: generated /Workspace/pipelines/sql/mx_claim.sql; run the pipeline to load devproduct.dev.ingestion_mx_claim
2026-10-25 15:05:23,585 | INFO | __main__ | Processed devproduct.dev.ingestion_mx_claim
